# CTD (Comparative Toxicogenomics Database) Preprocessing

This notebook processes CTD data into clean master and edge tables:

## Master Tables:
- `chemical_master_table` - Chemical/Drug entities
- `gene_master_table` - Gene entities
- `disease_master_table` - Disease entities
- `pathway_master_table` - Pathway entities

## Edge Tables:
- `chemical_gene_association` - Chemical-Gene interactions
- `chemical_disease_association` - Chemical-Disease associations
- `gene_disease_association` - Gene-Disease associations
- `gene_pathway_association` - Gene-Pathway associations
- `disease_pathway_association` - Disease-Pathway associations

**Process:** Load → Clean → Normalize → Deduplicate → Extract Masters → Combine → Save

## 1. Imports & Configuration

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
from pathlib import Path
from typing import List
import warnings

warnings.filterwarnings('ignore')

print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.3


In [2]:
# ===========================================================================
# CONFIGURATION
# ===========================================================================

DATA_DIR = Path("/home/abhishekh/abhi/external/ctd/")
OUTPUT_DIR = Path("/home/abhishekh/abhi/biochirp/database/ctd/")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# File paths
CTD_FILES = {
    "CHEM_GENE": {"path": DATA_DIR / "CTD_chem_gene_ixns.csv"},
    "CHEM_DISEASE": {"path": DATA_DIR / "CTD_curated_chemicals_diseases.csv"},
    "GENE_DISEASE": {"path": DATA_DIR / "CTD_curated_genes_diseases.csv"},
    "GENE_PATHWAY": {"path": DATA_DIR / "CTD_genes_pathways.csv"},
    "DISEASE_PATHWAY": {"path": DATA_DIR / "CTD_diseases_pathways.csv"},
    "CHEMICALS": {"path": DATA_DIR / "CTD_chemicals.csv"},
    "GENES": {"path": DATA_DIR / "CTD_genes.csv"},
    "DISEASES": {"path": DATA_DIR / "CTD_diseases.csv"},
    "PATHWAYS": {"path": DATA_DIR / "CTD_pathways.csv"},
}

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nFiles to process: {len(CTD_FILES)}")

Data directory: /home/abhishekh/abhi/external/ctd
Output directory: /home/abhishekh/abhi/biochirp/database/ctd

Files to process: 9


## 2. Helper Functions

In [3]:
# ===========================================================================
# HELPER FUNCTIONS
# ===========================================================================

def normalize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace from all string columns."""
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype("string").str.strip()
    return df


def enforce_id_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Convert ID columns to proper dtypes."""
    if 'gene_id' in df.columns:
        df['gene_id'] = pd.to_numeric(df['gene_id'], errors='coerce').astype('Int64')
    if 'drug_id' in df.columns:
        df['drug_id'] = df['drug_id'].astype('string')
    if 'disease_id' in df.columns:
        df['disease_id'] = df['disease_id'].astype('string')
    if 'pathway_id' in df.columns:
        df['pathway_id'] = df['pathway_id'].astype('string')
    return df


def drop_missing_required(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Drop rows with missing required columns."""
    cols = [c for c in cols if c in df.columns]
    return df.dropna(subset=cols) if cols else df


def deduplicate(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Remove duplicates based on key columns."""
    cols = [c for c in cols if c in df.columns]
    return df.drop_duplicates(subset=cols) if cols else df


def print_summary(name: str, df: pd.DataFrame, before: int = None):
    """Print processing summary."""
    if before:
        print(f"  {name}: {before:,} → {len(df):,} rows ({len(df)/before*100:.1f}% retained)")
    else:
        print(f"  {name}: {len(df):,} rows")


print("✓ Helper functions loaded")

✓ Helper functions loaded


## 3. Chemical-Gene Association

Processes chemical-gene interactions and extracts:
- Drug/Chemical master entries
- Gene master entries
- Chemical-Gene edge table

In [4]:
print("="*80)
print("PROCESSING: Chemical-Gene Association")
print("="*80)

# Load data
chemical_gene = pd.read_csv(
    CTD_FILES["CHEM_GENE"]["path"],
    skiprows=27,  # Skip header lines
    usecols=["# ChemicalName", "ChemicalID", "GeneSymbol", "GeneID", "Organism"]
)
initial_count = len(chemical_gene)
print(f"  Loaded: {initial_count:,} rows")

# Rename columns
chemical_gene = chemical_gene.rename(columns={
    '# ChemicalName': 'drug_name',
    'ChemicalID': 'drug_id',
    'CasRN': 'cas_rn',
    'GeneSymbol': 'gene_name',
    'GeneID': 'gene_id',
    'Organism': 'organism'
})

# Filter to Homo sapiens
chemical_gene = chemical_gene[chemical_gene["organism"] == "Homo sapiens"].copy()
print(f"  Filtered to human: {len(chemical_gene):,}/{initial_count:,} rows")

# Drop organism column (no longer needed)
chemical_gene = chemical_gene.drop(columns=["organism"])

# Normalize and enforce types
chemical_gene = normalize_strings(chemical_gene)
chemical_gene = enforce_id_dtypes(chemical_gene)

# Drop missing values
chemical_gene = drop_missing_required(chemical_gene, ['drug_name', 'drug_id', 'gene_name', 'gene_id'])

# Deduplicate
chemical_gene = deduplicate(chemical_gene, ['drug_id', 'gene_id'])
chemical_gene = chemical_gene.reset_index(drop=True)

print(f"  Final: {len(chemical_gene):,} rows")

# Extract drug master from chemical-gene
drug_master_from_chem_gene = (
    chemical_gene[['drug_id', 'drug_name']]
    .dropna(subset=['drug_id', 'drug_name'])
    .drop_duplicates(subset=['drug_id'])
    .sort_values('drug_id')
    .reset_index(drop=True)
)
print(f"  → Extracted drug master: {len(drug_master_from_chem_gene):,} unique drugs")

# Extract gene master from chemical-gene
gene_master_from_chem_gene = (
    chemical_gene[['gene_id', 'gene_name']]
    .dropna()
    .drop_duplicates(subset=['gene_id'])
    .sort_values('gene_id')
    .reset_index(drop=True)
)
print(f"  → Extracted gene master: {len(gene_master_from_chem_gene):,} unique genes")

# Create edge table (IDs only)
chemical_gene_association = (
    chemical_gene[['drug_id', 'gene_id']]
    .drop_duplicates()
    .sort_values(['drug_id', 'gene_id'])
    .reset_index(drop=True)
)
print(f"  → Edge table: {len(chemical_gene_association):,} associations")

print("\n✓ Chemical-Gene processing complete")
chemical_gene_association.head()

PROCESSING: Chemical-Gene Association
  Loaded: 2,983,668 rows
  Filtered to human: 1,293,600/2,983,668 rows
  Final: 809,922 rows
  → Extracted drug master: 10,951 unique drugs
  → Extracted gene master: 28,293 unique genes
  → Edge table: 809,922 associations

✓ Chemical-Gene processing complete


,drug_id,gene_id
0,C000015,5243
1,C000015,10257
2,C000061,501
3,C000081,6530
4,C000081,6531


## 4. Chemical-Disease Association

Processes chemical-disease associations and extracts:
- Drug/Chemical master entries
- Disease master entries
- Chemical-Disease edge table

In [5]:
print("="*80)
print("PROCESSING: Chemical-Disease Association")
print("="*80)

# Load data
chemical_disease = pd.read_csv(
    CTD_FILES["CHEM_DISEASE"]["path"],
    skiprows=27,
    usecols=["# ChemicalName", "ChemicalID", "DiseaseName", "DiseaseID"]
)
initial_count = len(chemical_disease)
print(f"  Loaded: {initial_count:,} rows")

# Rename columns
chemical_disease = chemical_disease.rename(columns={
    '# ChemicalName': 'drug_name',
    'ChemicalID': 'drug_id',
    'DiseaseName': 'disease_name',
    'DiseaseID': 'disease_id'
})
chemical_disease['disease_id'] = chemical_disease['disease_id'].str.replace("^MESH:", "", regex=True)


# Normalize and enforce types
chemical_disease = normalize_strings(chemical_disease)
chemical_disease = enforce_id_dtypes(chemical_disease)

# Drop missing values
chemical_disease = drop_missing_required(chemical_disease, ['drug_name', 'drug_id', 'disease_name', 'disease_id'])

# Deduplicate
chemical_disease = deduplicate(chemical_disease, ['drug_id', 'disease_id'])
chemical_disease = chemical_disease.reset_index(drop=True)

print_summary("After dedup", chemical_disease, initial_count)

# Extract drug master from chemical-disease
drug_master_from_chem_disease = (
    chemical_disease[['drug_id', 'drug_name']]
    .dropna()
    .drop_duplicates(subset=['drug_id'])
    .sort_values('drug_id')
    .reset_index(drop=True)
)
print(f"  → Extracted drug master: {len(drug_master_from_chem_disease):,} unique drugs")

# Extract disease master from chemical-disease
disease_master_from_chem_disease = (
    chemical_disease[['disease_id', 'disease_name']]
    .dropna()
    .drop_duplicates(subset=['disease_id'])
    .sort_values('disease_id')
    .reset_index(drop=True)
)
print(f"  → Extracted disease master: {len(disease_master_from_chem_disease):,} unique diseases")

# Create edge table (IDs only)
chemical_disease_association = (
    chemical_disease[['drug_id', 'disease_id']]
    .drop_duplicates()
    .sort_values(['drug_id', 'disease_id'])
    .reset_index(drop=True)
)
print(f"  → Edge table: {len(chemical_disease_association):,} associations")

print("\n✓ Chemical-Disease processing complete")
chemical_disease_association.head()

PROCESSING: Chemical-Disease Association
  Loaded: 108,441 rows
  After dedup: 108,441 → 104,628 rows (96.5% retained)
  → Extracted drug master: 10,639 unique drugs
  → Extracted disease master: 3,324 unique diseases
  → Edge table: 104,628 associations

✓ Chemical-Disease processing complete


,drug_id,disease_id
0,C000025,D004827
1,C000081,D006976
2,C000081,D012640
3,C000081,D019966
4,C000109,D001919


## 5. Gene-Pathway Association

Processes gene-pathway associations and extracts:
- Gene master entries
- Pathway master entries
- Gene-Pathway edge table

In [6]:
print("="*80)
print("PROCESSING: Gene-Pathway Association")
print("="*80)

# Load data
gene_pathway = pd.read_csv(
    CTD_FILES["GENE_PATHWAY"]["path"],
    skiprows=27,
    usecols=["# GeneSymbol", "GeneID", "PathwayName", "PathwayID"]
)
initial_count = len(gene_pathway)
print(f"  Loaded: {initial_count:,} rows")

# Rename columns
gene_pathway = gene_pathway.rename(columns={
    '# GeneSymbol': 'gene_name',
    'GeneID': 'gene_id',
    'PathwayName': 'pathway_name',
    'PathwayID': 'pathway_id'
})

# Clean pathway_id - remove REACT: prefix
gene_pathway['pathway_id'] = gene_pathway['pathway_id'].str.replace("^REACT:", "", regex=True)
# pathway_master_raw['pathway_id'] = pathway_master_raw['pathway_id'].str.replace("^REACT:", "", regex=True)
gene_pathway['pathway_id'] = gene_pathway['pathway_id'].str.replace("^KEGG:", "", regex=True)

# Normalize and enforce types
gene_pathway = normalize_strings(gene_pathway)
gene_pathway = enforce_id_dtypes(gene_pathway)

# Drop missing values
gene_pathway = drop_missing_required(gene_pathway, ['gene_name', 'gene_id', 'pathway_name', 'pathway_id'])

# Deduplicate
gene_pathway = deduplicate(gene_pathway, ['gene_id', 'pathway_id'])
gene_pathway = gene_pathway.reset_index(drop=True)

print_summary("After dedup", gene_pathway, initial_count)

# Extract gene master from gene-pathway
gene_master_from_gene_pathway = (
    gene_pathway[['gene_id', 'gene_name']]
    .dropna()
    .drop_duplicates(subset=['gene_id'])
    .sort_values('gene_id')
    .reset_index(drop=True)
)
print(f"  → Extracted gene master: {len(gene_master_from_gene_pathway):,} unique genes")

# Extract pathway master from gene-pathway
pathway_master_from_gene_pathway = (
    gene_pathway[['pathway_id', 'pathway_name']]
    .dropna()
    .drop_duplicates(subset=['pathway_id'])
    .sort_values('pathway_id')
    .reset_index(drop=True)
)
print(f"  → Extracted pathway master: {len(pathway_master_from_gene_pathway):,} unique pathways")

# Create edge table (IDs only)
gene_pathway_association = (
    gene_pathway[['gene_id', 'pathway_id']]
    .drop_duplicates()
    .sort_values(['gene_id', 'pathway_id'])
    .reset_index(drop=True)
)
print(f"  → Edge table: {len(gene_pathway_association):,} associations")

print("\n✓ Gene-Pathway processing complete")
gene_pathway_association.head()

PROCESSING: Gene-Pathway Association
  Loaded: 135,793 rows
  After dedup: 135,793 → 135,792 rows (100.0% retained)
  → Extracted gene master: 11,586 unique genes
  → Extracted pathway master: 2,363 unique pathways
  → Edge table: 135,792 associations

✓ Gene-Pathway processing complete


,gene_id,pathway_id
0,1,R-HSA-109582
1,1,R-HSA-114608
2,1,R-HSA-168249
3,1,R-HSA-168256
4,1,R-HSA-6798695


## 6. Disease-Pathway Association

Processes disease-pathway associations and extracts:
- Disease master entries
- Pathway master entries
- Disease-Pathway edge table

In [7]:
print("="*80)
print("PROCESSING: Disease-Pathway Association")
print("="*80)

# Load data
disease_pathway = pd.read_csv(
    CTD_FILES["DISEASE_PATHWAY"]["path"],
    skiprows=27,
    usecols=["# DiseaseName", "DiseaseID", "PathwayName", "PathwayID"]
)
initial_count = len(disease_pathway)
print(f"  Loaded: {initial_count:,} rows")

# Rename columns
disease_pathway = disease_pathway.rename(columns={
    '# DiseaseName': 'disease_name',
    'DiseaseID': 'disease_id',
    'PathwayName': 'pathway_name',
    'PathwayID': 'pathway_id'
})


# Clean pathway_id - remove REACT: and KEGG: prefixes
disease_pathway['pathway_id'] = disease_pathway['pathway_id'].str.replace("^REACT:", "", regex=True)
disease_pathway['pathway_id'] = disease_pathway['pathway_id'].str.replace("^KEGG:", "", regex=True)
disease_pathway['disease_id'] = disease_pathway['disease_id'].str.replace("^MESH:", "", regex=True)

# Normalize and enforce types
disease_pathway = normalize_strings(disease_pathway)
disease_pathway = enforce_id_dtypes(disease_pathway)

# Drop missing values
disease_pathway = drop_missing_required(disease_pathway, ['disease_name', 'disease_id', 'pathway_name', 'pathway_id'])

# Deduplicate
disease_pathway = deduplicate(disease_pathway, ['disease_id', 'pathway_id'])
disease_pathway = disease_pathway.reset_index(drop=True)

print_summary("After dedup", disease_pathway, initial_count)

# Extract disease master from disease-pathway
disease_master_from_disease_pathway = (
    disease_pathway[['disease_id', 'disease_name']]
    .dropna()
    .drop_duplicates(subset=['disease_id'])
    .sort_values('disease_id')
    .reset_index(drop=True)
)
print(f"  → Extracted disease master: {len(disease_master_from_disease_pathway):,} unique diseases")

# Extract pathway master from disease-pathway
pathway_master_from_disease_pathway = (
    disease_pathway[['pathway_id', 'pathway_name']]
    .dropna()
    .drop_duplicates(subset=['pathway_id'])
    .sort_values('pathway_id')
    .reset_index(drop=True)
)
print(f"  → Extracted pathway master: {len(pathway_master_from_disease_pathway):,} unique pathways")

# Create edge table (IDs only)
disease_pathway_association = (
    disease_pathway[['disease_id', 'pathway_id']]
    .drop_duplicates()
    .sort_values(['disease_id', 'pathway_id'])
    .reset_index(drop=True)
)
print(f"  → Edge table: {len(disease_pathway_association):,} associations")

print("\n✓ Disease-Pathway processing complete")
disease_pathway_association.head()

PROCESSING: Disease-Pathway Association
  Loaded: 616,230 rows
  After dedup: 616,230 → 306,745 rows (49.8% retained)
  → Extracted disease master: 5,052 unique diseases
  → Extracted pathway master: 2,339 unique pathways
  → Edge table: 306,745 associations

✓ Disease-Pathway processing complete


,disease_id,pathway_id
0,C000600608,R-HSA-1430728
1,C000600608,R-HSA-1630316
2,C000600608,R-HSA-1793185
3,C000600608,R-HSA-2022923
4,C000600608,R-HSA-71387


## 7. Gene-Disease Association

Processes gene-disease associations and extracts:
- Gene master entries
- Disease master entries
- Gene-Disease edge table

In [8]:
print("="*80)
print("PROCESSING: Gene-Disease Association")
print("="*80)

# Load data
gene_disease = pd.read_csv(
    CTD_FILES["GENE_DISEASE"]["path"],
    skiprows=27,
    usecols=["# GeneSymbol", "GeneID", "DiseaseName", "DiseaseID"]
)
initial_count = len(gene_disease)
print(f"  Loaded: {initial_count:,} rows")

# Rename columns
gene_disease = gene_disease.rename(columns={
    '# GeneSymbol': 'gene_name',
    'GeneID': 'gene_id',
    'DiseaseName': 'disease_name',
    'DiseaseID': 'disease_id'
})

gene_disease['disease_id'] = gene_disease['disease_id'].str.replace("^MESH:", "", regex=True)


# Normalize and enforce types
gene_disease = normalize_strings(gene_disease)
gene_disease = enforce_id_dtypes(gene_disease)

# Drop missing values
gene_disease = drop_missing_required(gene_disease, ['gene_name', 'gene_id', 'disease_name', 'disease_id'])

# Deduplicate
gene_disease = deduplicate(gene_disease, ['gene_id', 'disease_id'])
gene_disease = gene_disease.reset_index(drop=True)

print_summary("After dedup", gene_disease, initial_count)

# Extract gene master from gene-disease
gene_master_from_gene_disease = (
    gene_disease[['gene_id', 'gene_name']]
    .dropna()
    .drop_duplicates(subset=['gene_id'])
    .sort_values('gene_id')
    .reset_index(drop=True)
)
print(f"  → Extracted gene master: {len(gene_master_from_gene_disease):,} unique genes")

# Extract disease master from gene-disease
disease_master_from_gene_disease = (
    gene_disease[['disease_id', 'disease_name']]
    .dropna()
    .drop_duplicates(subset=['disease_id'])
    .sort_values('disease_id')
    .reset_index(drop=True)
)
print(f"  → Extracted disease master: {len(disease_master_from_gene_disease):,} unique diseases")

# Create edge table (IDs only)
gene_disease_association = (
    gene_disease[['gene_id', 'disease_id']]
    .drop_duplicates()
    .sort_values(['gene_id', 'disease_id'])
    .reset_index(drop=True)
)
print(f"  → Edge table: {len(gene_disease_association):,} associations")

print("\n✓ Gene-Disease processing complete")
gene_disease_association.head()

PROCESSING: Gene-Disease Association
  Loaded: 35,430 rows
  After dedup: 35,430 → 35,429 rows (100.0% retained)
  → Extracted gene master: 9,209 unique genes
  → Extracted disease master: 5,855 unique diseases
  → Edge table: 35,429 associations

✓ Gene-Disease processing complete


,gene_id,disease_id
0,1,D006529
1,1,D012559
2,2,D000544
3,2,D003110
4,2,D006527


## 8. Load Raw Master Tables

Load the official CTD master tables for chemicals, genes, diseases, and pathways.

In [9]:
print("="*80)
print("LOADING: Chemical Master Table (Raw)")
print("="*80)

# Load chemical master
chemical_master_raw = pd.read_csv(
    CTD_FILES['CHEMICALS']['path'],
    skiprows=27,
    usecols=["# ChemicalName", "ChemicalID"]
)
print(f"  Loaded: {len(chemical_master_raw):,} rows")

# Rename columns
chemical_master_raw = chemical_master_raw.rename(columns={
    '# ChemicalName': 'drug_name',
    'ChemicalID': 'drug_id',
    'CasRN': 'cas_rn'
})

# Clean drug_id - remove MESH: prefix
chemical_master_raw['drug_id'] = chemical_master_raw['drug_id'].str.replace("^MESH:", "", regex=True)

# Normalize
chemical_master_raw = normalize_strings(chemical_master_raw)
chemical_master_raw = drop_missing_required(chemical_master_raw, ['drug_id', 'drug_name'])
chemical_master_raw = deduplicate(chemical_master_raw, ['drug_id'])
chemical_master_raw = chemical_master_raw.sort_values('drug_id').reset_index(drop=True)

print(f"  Final: {len(chemical_master_raw):,} rows")
print("✓ Chemical master loaded")
chemical_master_raw.head()

LOADING: Chemical Master Table (Raw)
  Loaded: 179,407 rows
  Final: 179,406 rows
✓ Chemical master loaded


,drug_name,drug_id
0,bevonium,C000002
1,"insulin, neutral",C000006
2,N-acetylglucosaminylasparagine,C000009
3,5-(n-acetaminophenylazo)-8-oxyquinoline,C000011
4,N-acetyl-L-arginine,C000015


In [10]:
print("="*80)
print("LOADING: Gene Master Table (Raw)")
print("="*80)

# Load gene master
gene_master_raw = pd.read_csv(
    CTD_FILES["GENES"]['path'],
    skiprows=27,  # Skip 27 header + 2 data rows
    usecols=["# GeneSymbol", "GeneID"]
)
print(f"  Loaded: {len(gene_master_raw):,} rows")

# Rename columns
gene_master_raw = gene_master_raw.rename(columns={
    '# GeneSymbol': 'gene_name',
    'GeneID': 'gene_id',
    'PharmGKBIDs': 'pharm_gkb_ids'
})

# Normalize and enforce types
gene_master_raw = normalize_strings(gene_master_raw)
gene_master_raw = enforce_id_dtypes(gene_master_raw)
gene_master_raw = drop_missing_required(gene_master_raw, ['gene_id', 'gene_name'])
gene_master_raw = deduplicate(gene_master_raw, ['gene_id'])
gene_master_raw = gene_master_raw.sort_values('gene_id').reset_index(drop=True)

print(f"  Final: {len(gene_master_raw):,} rows")
print("✓ Gene master loaded")
gene_master_raw.head()

LOADING: Gene Master Table (Raw)
  Loaded: 638,839 rows
  Final: 638,837 rows
✓ Gene master loaded


,gene_name,gene_id
0,A1BG,1
1,A2M,2
2,NAT1,9
3,NAT2,10
4,NATP,11


In [11]:
print("="*80)
print("LOADING: Disease Master Table (Raw)")
print("="*80)

# Load disease master
disease_master_raw = pd.read_csv(
    CTD_FILES["DISEASES"]['path'],
    skiprows=27,
    usecols=["# DiseaseName", "DiseaseID"]
)

print(f"  Loaded: {len(disease_master_raw):,} rows")

# Rename columns
disease_master_raw = disease_master_raw.rename(columns={
    '# DiseaseName': 'disease_name',
    'DiseaseID': 'disease_id'
})

disease_master_raw = disease_master_raw[disease_master_raw["disease_name"]!="Diseases"]

# Clean disease_id - remove MESH: prefix
disease_master_raw['disease_id'] = disease_master_raw['disease_id'].str.replace("^MESH:", "", regex=True)

# Normalize
disease_master_raw = normalize_strings(disease_master_raw)
disease_master_raw = drop_missing_required(disease_master_raw, ['disease_id', 'disease_name'])
disease_master_raw = deduplicate(disease_master_raw, ['disease_id'])
disease_master_raw = disease_master_raw.sort_values('disease_id').reset_index(drop=True)

print(f"  Final: {len(disease_master_raw):,} rows")
print("✓ Disease master loaded")
disease_master_raw.head()

LOADING: Disease Master Table (Raw)
  Loaded: 13,318 rows
  Final: 13,316 rows
✓ Disease master loaded


,disease_name,disease_id
0,"familial gynecomastia, due to increased aromat...",C000591739
1,Jalili syndrome,C000596385
2,Typical Teratoid Rhabdoid Tumor,C000597569
3,"Retinoschisis, Autosomal Dominant",C000598640
4,Leukoencephalopathy Brain Calcifications and C...,C000598644


In [12]:
print("="*80)
print("LOADING: Pathway Master Table (Raw)")
print("="*80)

# Load pathway master
pathway_master_raw = pd.read_csv(
    CTD_FILES['PATHWAYS']['path'],
    skiprows=27,
    usecols=["# PathwayName", "PathwayID"]
)
print(f"  Loaded: {len(pathway_master_raw):,} rows")

# Rename columns
pathway_master_raw = pathway_master_raw.rename(columns={
    '# PathwayName': 'pathway_name',
    'PathwayID': 'pathway_id'
})

# Clean pathway_id - remove REACT: and KEGG: prefixes
pathway_master_raw['pathway_id'] = pathway_master_raw['pathway_id'].str.replace("^REACT:", "", regex=True)
pathway_master_raw['pathway_id'] = pathway_master_raw['pathway_id'].str.replace("^KEGG:", "", regex=True)

# Normalize
pathway_master_raw = normalize_strings(pathway_master_raw)
pathway_master_raw = drop_missing_required(pathway_master_raw, ['pathway_id', 'pathway_name'])
pathway_master_raw = deduplicate(pathway_master_raw, ['pathway_id'])
pathway_master_raw = pathway_master_raw.sort_values('pathway_id').reset_index(drop=True)

print(f"  Final: {len(pathway_master_raw):,} rows")
print("✓ Pathway master loaded")
pathway_master_raw.head()

LOADING: Pathway Master Table (Raw)
  Loaded: 2,568 rows


  Final: 2,567 rows
✓ Pathway master loaded


,pathway_name,pathway_id
0,Interleukin-6 signaling,R-HSA-1059683
1,Apoptosis,R-HSA-109581
2,Hemostasis,R-HSA-109582
3,Intrinsic Pathway for Apoptosis,R-HSA-109606
4,Cleavage of Growing Transcript in the Terminat...,R-HSA-109688


## 8. Combine Master Tables

Combine all sources (raw files + extracted from edges) to create comprehensive master tables.

In [13]:
print("="*80)
print("COMBINING: Chemical Master Table")
print("="*80)

# Combine all chemical/drug sources
print("Sources:")
print(f"  - Raw file: {len(chemical_master_raw):,} rows")
print(f"  - From chemical-gene: {len(drug_master_from_chem_gene):,} rows")
print(f"  - From chemical-disease: {len(drug_master_from_chem_disease):,} rows")

chemical_master_table = pd.concat([
    chemical_master_raw,
    drug_master_from_chem_gene,
    drug_master_from_chem_disease
], ignore_index=True)

print(f"\nBefore dedup: {len(chemical_master_table):,} rows")

# Deduplicate on drug_id (keep first occurrence)
chemical_master_table = chemical_master_table.drop_duplicates(subset=['drug_id'])
chemical_master_table = chemical_master_table.sort_values('drug_id').reset_index(drop=True)

print(f"After dedup: {len(chemical_master_table):,} rows")
print(f"\n✓ Combined chemical master table created")
print(f"Columns: {list(chemical_master_table.columns)}")
chemical_master_table.head()

COMBINING: Chemical Master Table
Sources:
  - Raw file: 179,406 rows
  - From chemical-gene: 10,951 rows
  - From chemical-disease: 10,639 rows

Before dedup: 200,996 rows
After dedup: 179,406 rows

✓ Combined chemical master table created
Columns: ['drug_name', 'drug_id']


,drug_name,drug_id
0,bevonium,C000002
1,"insulin, neutral",C000006
2,N-acetylglucosaminylasparagine,C000009
3,5-(n-acetaminophenylazo)-8-oxyquinoline,C000011
4,N-acetyl-L-arginine,C000015


In [14]:
print("="*80)
print("COMBINING: Gene Master Table")
print("="*80)

# Combine all gene sources
print("Sources:")
print(f"  - Raw file: {len(gene_master_raw):,} rows")
print(f"  - From chemical-gene: {len(gene_master_from_chem_gene):,} rows")
print(f"  - From gene-disease: {len(gene_master_from_gene_disease):,} rows")
print(f"  - From gene-pathway: {len(gene_master_from_gene_pathway):,} rows")

gene_master_table = pd.concat([
    gene_master_raw,
    gene_master_from_chem_gene[['gene_id', 'gene_name']],  # Align columns
    gene_master_from_gene_disease,
    gene_master_from_gene_pathway
], ignore_index=True)

print(f"\nBefore dedup: {len(gene_master_table):,} rows")

# Deduplicate on gene_id (keep first occurrence)
gene_master_table = gene_master_table.drop_duplicates(subset=['gene_id'])
gene_master_table = gene_master_table.sort_values('gene_id').reset_index(drop=True)

print(f"After dedup: {len(gene_master_table):,} rows")
print(f"\n✓ Combined gene master table created")
print(f"Columns: {list(gene_master_table.columns)}")
gene_master_table.head()

COMBINING: Gene Master Table
Sources:
  - Raw file: 638,837 rows
  - From chemical-gene: 28,293 rows
  - From gene-disease: 9,209 rows
  - From gene-pathway: 11,586 rows

Before dedup: 687,925 rows
After dedup: 638,837 rows

✓ Combined gene master table created
Columns: ['gene_name', 'gene_id']


,gene_name,gene_id
0,A1BG,1
1,A2M,2
2,NAT1,9
3,NAT2,10
4,NATP,11


In [15]:
print("="*80)
print("COMBINING: Disease Master Table")
print("="*80)

# Combine all disease sources
print("Sources:")
print(f"  - Raw file: {len(disease_master_raw):,} rows")
print(f"  - From chemical-disease: {len(disease_master_from_chem_disease):,} rows")
print(f"  - From disease-pathway: {len(disease_master_from_disease_pathway):,} rows")

disease_master_table = pd.concat([
    disease_master_raw,
    disease_master_from_chem_disease,
    disease_master_from_disease_pathway
], ignore_index=True)

print(f"\nBefore dedup: {len(disease_master_table):,} rows")

# Deduplicate on disease_id (keep first occurrence)
disease_master_table = disease_master_table.drop_duplicates(subset=['disease_id'])
disease_master_table = disease_master_table.sort_values('disease_id').reset_index(drop=True)

print(f"After dedup: {len(disease_master_table):,} rows")
print(f"\n✓ Combined disease master table created")
print(f"Columns: {list(disease_master_table.columns)}")
disease_master_table.head()

COMBINING: Disease Master Table
Sources:
  - Raw file: 13,316 rows
  - From chemical-disease: 3,324 rows
  - From disease-pathway: 5,052 rows

Before dedup: 21,692 rows
After dedup: 13,316 rows

✓ Combined disease master table created
Columns: ['disease_name', 'disease_id']


,disease_name,disease_id
0,"familial gynecomastia, due to increased aromat...",C000591739
1,Jalili syndrome,C000596385
2,Typical Teratoid Rhabdoid Tumor,C000597569
3,"Retinoschisis, Autosomal Dominant",C000598640
4,Leukoencephalopathy Brain Calcifications and C...,C000598644


In [16]:
print("="*80)
print("COMBINING: Pathway Master Table")
print("="*80)

# Combine all pathway sources
print("Sources:")
print(f"  - Raw file: {len(pathway_master_raw):,} rows")
print(f"  - From gene-pathway: {len(pathway_master_from_gene_pathway):,} rows")
print(f"  - From disease-pathway: {len(pathway_master_from_disease_pathway):,} rows")

pathway_master_table = pd.concat([
    pathway_master_raw,
    pathway_master_from_gene_pathway,
    pathway_master_from_disease_pathway
], ignore_index=True)

print(f"\nBefore dedup: {len(pathway_master_table):,} rows")

# Deduplicate on pathway_id (keep first occurrence)
pathway_master_table = pathway_master_table.drop_duplicates(subset=['pathway_id'])
pathway_master_table = pathway_master_table.sort_values('pathway_id').reset_index(drop=True)

print(f"After dedup: {len(pathway_master_table):,} rows")
print(f"\n✓ Combined pathway master table created")
print(f"Columns: {list(pathway_master_table.columns)}")
pathway_master_table.head()

COMBINING: Pathway Master Table
Sources:
  - Raw file: 2,567 rows
  - From gene-pathway: 2,363 rows
  - From disease-pathway: 2,339 rows

Before dedup: 7,269 rows
After dedup: 2,567 rows

✓ Combined pathway master table created
Columns: ['pathway_name', 'pathway_id']


,pathway_name,pathway_id
0,Interleukin-6 signaling,R-HSA-1059683
1,Apoptosis,R-HSA-109581
2,Hemostasis,R-HSA-109582
3,Intrinsic Pathway for Apoptosis,R-HSA-109606
4,Cleavage of Growing Transcript in the Terminat...,R-HSA-109688


## 9. Save All Tables

Save all master and edge tables to parquet format.

In [17]:
# ===============================
# DROP ROWS WITH '' OR '.' IN ANY CELL
# ===============================

def drop_empty_or_dot_rows(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.apply(lambda col: col.astype(str).isin(["", "."]), axis=0).any(axis=1)
    return df.loc[~mask].reset_index(drop=True)

# Apply to final tables before saving

chemical_master_table = drop_empty_or_dot_rows(chemical_master_table)
gene_master_table = drop_empty_or_dot_rows(gene_master_table)
disease_master_table = drop_empty_or_dot_rows(disease_master_table)
pathway_master_table = drop_empty_or_dot_rows(pathway_master_table)

chemical_gene_association = drop_empty_or_dot_rows(chemical_gene_association)
chemical_disease_association = drop_empty_or_dot_rows(chemical_disease_association)
gene_disease_association = drop_empty_or_dot_rows(gene_disease_association)
gene_pathway_association = drop_empty_or_dot_rows(gene_pathway_association)
disease_pathway_association = drop_empty_or_dot_rows(disease_pathway_association)


In [18]:
print("="*80)
print("SAVING TABLES")
print("="*80)

# Save master tables
print("\nMaster Tables:")
# Fail fast: if any cell contains ';', clean before saving
if chemical_master_table.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    chemical_master_table = chemical_master_table.replace({r";.*$": ""}, regex=True)

chemical_master_table.to_parquet(OUTPUT_DIR / "chemical_master_table.parquet", index=False)
print(f"  ✓ chemical_master_table.parquet: {len(chemical_master_table):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if gene_master_table.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    gene_master_table = gene_master_table.replace({r";.*$": ""}, regex=True)

gene_master_table.to_parquet(OUTPUT_DIR / "gene_master_table.parquet", index=False)
print(f"  ✓ gene_master_table.parquet: {len(gene_master_table):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if disease_master_table.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    disease_master_table = disease_master_table.replace({r";.*$": ""}, regex=True)

disease_master_table.to_parquet(OUTPUT_DIR / "disease_master_table.parquet", index=False)
print(f"  ✓ disease_master_table.parquet: {len(disease_master_table):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if pathway_master_table.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    pathway_master_table = pathway_master_table.replace({r";.*$": ""}, regex=True)

pathway_master_table.to_parquet(OUTPUT_DIR / "pathway_master_table.parquet", index=False)
print(f"  ✓ pathway_master_table.parquet: {len(pathway_master_table):,} rows")

# Save edge tables
print("\nEdge Tables:")
# Fail fast: if any cell contains ';', clean before saving
if chemical_gene_association.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    chemical_gene_association = chemical_gene_association.replace({r";.*$": ""}, regex=True)

chemical_gene_association.to_parquet(OUTPUT_DIR / "chem_gene_association.parquet", index=False)
print(f"  ✓ chemical_gene_association.parquet: {len(chemical_gene_association):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if chemical_disease_association.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    chemical_disease_association = chemical_disease_association.replace({r";.*$": ""}, regex=True)

chemical_disease_association.to_parquet(OUTPUT_DIR / "chemical_disease_association.parquet", index=False)
print(f"  ✓ chemical_disease_association.parquet: {len(chemical_disease_association):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if gene_disease_association.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    gene_disease_association = gene_disease_association.replace({r";.*$": ""}, regex=True)

gene_disease_association.to_parquet(OUTPUT_DIR / "gene_disease_association.parquet", index=False)
print(f"  ✓ gene_disease_association.parquet: {len(gene_disease_association):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if gene_pathway_association.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    gene_pathway_association = gene_pathway_association.replace({r";.*$": ""}, regex=True)

gene_pathway_association.to_parquet(OUTPUT_DIR / "gene_pathway_association.parquet", index=False)
print(f"  ✓ gene_pathway_association.parquet: {len(gene_pathway_association):,} rows")

# Fail fast: if any cell contains ';', clean before saving
if disease_pathway_association.astype(str).apply(lambda s: s.str.contains(';', regex=False, na=False)).any().any():
    disease_pathway_association = disease_pathway_association.replace({r";.*$": ""}, regex=True)

disease_pathway_association.to_parquet(OUTPUT_DIR / "disease_pathway_association.parquet", index=False)
print(f"  ✓ disease_pathway_association.parquet: {len(disease_pathway_association):,} rows")

print("\n" + "="*80)
print("✅ ALL TABLES SAVED SUCCESSFULLY!")
print("="*80)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Total files: 9 (4 master + 5 edge)")

SAVING TABLES

Master Tables:
  ✓ chemical_master_table.parquet: 179,406 rows
  ✓ gene_master_table.parquet: 638,837 rows
  ✓ disease_master_table.parquet: 13,316 rows
  ✓ pathway_master_table.parquet: 2,567 rows

Edge Tables:
  ✓ chemical_gene_association.parquet: 809,922 rows
  ✓ chemical_disease_association.parquet: 104,628 rows
  ✓ gene_disease_association.parquet: 35,429 rows
  ✓ gene_pathway_association.parquet: 135,792 rows
  ✓ disease_pathway_association.parquet: 306,745 rows

✅ ALL TABLES SAVED SUCCESSFULLY!

Output directory: /home/abhishekh/abhi/biochirp/database/ctd
Total files: 9 (4 master + 5 edge)


## 10. Final Summary

Display final statistics for all tables.

In [19]:
print("="*80)
print("Comparative Toxicogenomics Database SUMMARY")
print("="*80)

print("\n📊 MASTER TABLES:")
print(f"  • chemical_master_table: {len(chemical_master_table):,} rows")
print(f"    Columns: {list(chemical_master_table.columns)}")
print(f"\n  • gene_master_table: {len(gene_master_table):,} rows")
print(f"    Columns: {list(gene_master_table.columns)}")
print(f"\n  • disease_master_table: {len(disease_master_table):,} rows")
print(f"    Columns: {list(disease_master_table.columns)}")
print(f"\n  • pathway_master_table: {len(pathway_master_table):,} rows")
print(f"    Columns: {list(pathway_master_table.columns)}")

print("\n🔗 EDGE TABLES:")
print(f"  • chemical_gene_association: {len(chemical_gene_association):,} rows")
print(f"    Columns: {list(chemical_gene_association.columns)}")
print(f"\n  • chemical_disease_association: {len(chemical_disease_association):,} rows")
print(f"    Columns: {list(chemical_disease_association.columns)}")
print(f"\n  • gene_disease_association: {len(gene_disease_association):,} rows")
print(f"    Columns: {list(gene_disease_association.columns)}")
print(f"\n  • gene_pathway_association: {len(gene_pathway_association):,} rows")
print(f"    Columns: {list(gene_pathway_association.columns)}")
print(f"\n  • disease_pathway_association: {len(disease_pathway_association):,} rows")
print(f"    Columns: {list(disease_pathway_association.columns)}")

print(f"\nNumber of unique chemical name: {chemical_master_table['drug_name'].nunique() if 'drug_name' in chemical_master_table.columns else 'N/A'}")
print(f"Number of unique gene name: {gene_master_table['gene_name'].nunique() if 'gene_name' in gene_master_table.columns else 'N/A'}")
print(f"Number of unique disease name: {disease_master_table['disease_name'].nunique() if 'disease_name' in disease_master_table.columns else 'N/A'}")
print(f"Number of unique pathway name: {pathway_master_table['pathway_name'].nunique() if 'pathway_name' in pathway_master_table.columns else 'N/A'}")

print("\n" + "="*80)
print("✅ CTD PREPROCESSING COMPLETE!")
print("="*80)

Comparative Toxicogenomics Database SUMMARY

📊 MASTER TABLES:
  • chemical_master_table: 179,406 rows
    Columns: ['drug_name', 'drug_id']

  • gene_master_table: 638,837 rows
    Columns: ['gene_name', 'gene_id']

  • disease_master_table: 13,316 rows
    Columns: ['disease_name', 'disease_id']

  • pathway_master_table: 2,567 rows
    Columns: ['pathway_name', 'pathway_id']

🔗 EDGE TABLES:
  • chemical_gene_association: 809,922 rows
    Columns: ['drug_id', 'gene_id']

  • chemical_disease_association: 104,628 rows
    Columns: ['drug_id', 'disease_id']

  • gene_disease_association: 35,429 rows
    Columns: ['gene_id', 'disease_id']

  • gene_pathway_association: 135,792 rows
    Columns: ['gene_id', 'pathway_id']

  • disease_pathway_association: 306,745 rows
    Columns: ['disease_id', 'pathway_id']

Number of unique chemical name: 179406
Number of unique gene name: 638698
Number of unique disease name: 13316
Number of unique pathway name: 2545

✅ CTD PREPROCESSING COMPLETE!
